# 09 — Modelo de Deep Learning (MLP em PyTorch)
Entrada: 17.955 atributos de FC. Um conjunto de validação (20% do treino) controla o *early stopping*; o teste só é avaliado ao final.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import joblib, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, f1_score
import config
from nn_utils import MLP

dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(config.SEED)
print("Dispositivo:", dev)

d = np.load(config.ARQ_DATASET, allow_pickle=True)
X_all, y_all, Xte, yte = d["X_train"], d["y_train"], d["X_test"], d["y_test"]
Xtr, Xva, ytr, yva = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=config.SEED)

sc = StandardScaler().fit(Xtr)
T = lambda a: torch.tensor(a, dtype=torch.float32, device=dev)
Xtr_t, Xva_t, Xte_t = T(sc.transform(Xtr)), T(sc.transform(Xva)), T(sc.transform(Xte))
ytr_t = T(ytr)
print(Xtr_t.shape, Xva_t.shape, Xte_t.shape)

In [ ]:
model = MLP(Xtr_t.shape[1]).to(dev)
pos_weight = torch.tensor((ytr == 0).sum() / (ytr == 1).sum(), dtype=torch.float32, device=dev)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

EPOCAS, BATCH, PACIENCIA = 200, 32, 25
hist = {"loss": [], "auc_val": []}
melhor_auc, melhor_estado, sem_melhora = -1, None, 0
g = torch.Generator().manual_seed(config.SEED)

for ep in range(EPOCAS):
    model.train()
    perm = torch.randperm(len(Xtr_t), generator=g)
    total = 0.0
    for i in range(0, len(perm), BATCH):
        b = perm[i:i + BATCH]
        if len(b) < 2:
            continue
        opt.zero_grad()
        loss = loss_fn(model(Xtr_t[b]), ytr_t[b])
        loss.backward(); opt.step()
        total += loss.item() * len(b)

    model.eval()
    with torch.no_grad():
        pv = torch.sigmoid(model(Xva_t)).cpu().numpy()
    auc = roc_auc_score(yva, pv)
    hist["loss"].append(total / len(Xtr_t)); hist["auc_val"].append(auc)

    if auc > melhor_auc:
        melhor_auc, sem_melhora = auc, 0
        melhor_estado = {k: v.detach().clone() for k, v in model.state_dict().items()}
    else:
        sem_melhora += 1
    if ep % 10 == 0:
        print(f"ep {ep:3d} | loss {hist['loss'][-1]:.4f} | AUC val {auc:.3f}")
    if sem_melhora >= PACIENCIA:
        print(f"Early stopping na época {ep}. Melhor AUC val: {melhor_auc:.3f}")
        break

model.load_state_dict(melhor_estado)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist["loss"]); ax[0].set_title("Loss de treino")
ax[1].plot(hist["auc_val"]); ax[1].set_title("AUC de validação")
plt.tight_layout(); plt.savefig(config.FIGURAS / "09_curvas_treino.png", dpi=150); plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    p_te = torch.sigmoid(model(Xte_t)).cpu().numpy()
pred = (p_te >= 0.5).astype(int)
print(f"TESTE | AUC {roc_auc_score(yte, p_te):.3f} | BAcc {balanced_accuracy_score(yte, pred):.3f} "
      f"| Acc {accuracy_score(yte, pred):.3f} | F1 {f1_score(yte, pred):.3f}")

torch.save(model.state_dict(), config.MODELOS / "mlp.pt")
joblib.dump(sc, config.MODELOS / "scaler_mlp.joblib")
np.save(config.RESULTADOS / "proba_mlp.npy", p_te)